## Tutorial:
This notebook is designed to walk you through a simple use case of GTEx-Tissue-Classifier. 

### Requirements:
- RNA Sequencing file of sample to test, with Ensembl IDs as row names and TPM values as entries in columns. 


In [6]:
import gtex_tissue_classifier
from gtex_tissue_classifier import architecture, math_logic, real_data, visualization, simulation
from gtex_tissue_classifier.validation import InputValidation

We begin by generating sample input data, both an example RNA sequencing file and a mock GTEx tissue data file. num_genes and num_tissues can be adjusted depending on the desired size of the sample dataset. 

In [7]:
simulation.generate_mock_tissues('sample_gtex.gct', num_genes=100, num_tissues=50)
simulation.generate_mock_sample('sample_tissues.csv', num_genes=100)

Sample tissue matrix created at sample_gtex.gct
Mock sample created and saved to sample_tissues.csv


In [8]:
gct = real_data.parse_raw_gct('sample_gtex.gct')
real_data.compute_TPM(gct)
gct

,GTEX-0,GTEX-1,GTEX-2,GTEX-3,GTEX-4,GTEX-5,GTEX-6,GTEX-7,GTEX-8,GTEX-9,GTEX-10,GTEX-11,GTEX-12,GTEX-13,GTEX-14,GTEX-15,GTEX-16,GTEX-17,GTEX-18,GTEX-19,GTEX-20,GTEX-21,GTEX-22,GTEX-23,GTEX-24,GTEX-25,GTEX-26,GTEX-27,GTEX-28,GTEX-29,GTEX-30,GTEX-31,GTEX-32,GTEX-33,GTEX-34,GTEX-35,GTEX-36,GTEX-37,GTEX-38,GTEX-39,GTEX-40,GTEX-41,GTEX-42,GTEX-43,GTEX-44,GTEX-45,GTEX-46,GTEX-47,GTEX-48,GTEX-49,Median_TPM
Name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
ENSG000000,0.39414,0.74766,1.03962,2.42382,0.17772,0.41165,3.31976,0.45656,1.07559,0.68614,2.12846,0.49028,0.61305,0.59664,0.13584,0.24916,0.35219,0.27142,0.26433,0.21317,0.52182,0.35376,0.80701,0.40102,1.79042,0.33762,0.43679,0.19288,1.13307,0.36799,0.10595,0.13285,1.02201,0.14956,1.46870,0.10400,0.04045,0.19106,0.43241,0.20046,0.19170,0.26778,0.11533,1.84527,0.06257,0.58430,3.43401,0.24460,2.08079,0.53281,0.406335
ENSG000001,0.38743,0.32434,0.60968,0.57048,0.05801,0.41804,0.43724,0.20286,0.39960,0.21763,0.42302,2.79387,2.13917,0.19159,0.33027,0.06135,0.01407,1.43418,0.34703,0.10439,0.98053,0.37926,0.25707,0.09859,2.80935,0.66392,0.24013,0.76059,0.76072,0.32836,0.10212,0.21429,1.52663,0.08265,0.72028,3.73591,0.86294,1.54432,0.16279,0.54690,4.65010,0.97568,0.09393,0.24699,0.30541,0.10640,0.56312,0.41879,0.21408,0.79663,0.393515
ENSG000002,0.24999,0.24001,0.19704,1.76955,0.26971,0.22351,0.56097,0.16008,0.36680,0.32322,0.60914,0.18640,0.53228,0.74934,0.24023,0.76554,0.63844,0.61358,0.09070,0.55880,0.07972,1.21552,0.52215,0.31178,0.06682,0.18051,0.43064,7.09395,0.23468,0.19758,0.88718,0.53942,1.31357,0.47139,1.37031,0.10228,0.84253,0.04047,0.22892,0.81854,0.35752,0.03562,2.56322,0.49780,0.06377,0.49707,2.16048,0.88246,0.54615,0.14422,0.451015
ENSG000003,0.14459,0.27252,0.18069,0.24681,0.37760,0.19359,0.12496,0.62868,0.64844,0.15420,1.36370,0.77696,0.06590,0.70811,0.20584,0.82701,0.71205,0.37532,0.61238,0.52605,2.16510,0.79038,0.12212,0.10928,0.40561,0.55590,0.13099,0.60106,0.07838,0.06700,0.94604,0.15814,0.50950,0.69104,0.19454,0.60325,1.73557,0.29952,0.29467,0.47093,0.14799,0.14894,0.14043,0.11304,0.25732,0.65407,0.28815,0.13311,0.34928,0.14818,0.297095
ENSG000004,0.97299,0.99102,0.13602,0.80722,0.77266,0.58992,0.61788,0.64845,0.19535,0.74983,0.53024,0.43689,0.35418,0.11183,1.63259,0.39455,0.22664,0.38841,0.44761,0.76244,0.64410,1.17582,0.72245,0.47741,0.30212,1.18051,0.22553,0.20032,0.52039,0.36719,0.76622,1.24508,0.48322,0.28156,0.19104,0.10316,2.44783,0.10308,0.56428,0.73047,0.67079,0.19140,0.36574,1.66877,0.97548,1.79333,0.28527,0.17620,1.16757,0.15259,0.525315
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG0000095,2.16984,0.35347,0.05172,2.07275,0.13840,0.27202,0.34136,0.42179,0.22152,0.13197,1.69808,2.36632,0.30586,0.40582,0.75648,1.08177,0.20551,0.10959,0.57106,0.70811,1.02175,0.83426,0.79414,0.12194,0.29250,0.02165,0.97397,0.19172,1.22248,0.23074,1.49553,0.10788,0.13193,0.38608,0.07524,0.82031,0.38832,0.44133,0.13502,0.07423,0.90880,0.41585,0.15661,0.15959,0.25017,0.29029,0.43520,0.12734,0.02254,1.08677,0.347415
ENSG0000096,0.98332,0.09234,0.20909,0.02837,1.02023,0.66471,0.19593,1.13277,0.63496,1.19653,0.38191,0.97142,0.09020,1.09700,0.82423,0.30847,0.29908,0.03533,0.37141,0.57415,0.28510,0.53793,0.23869,0.75505,1.52899,0.16805,0.28857,0.72686,0.25654,0.26922,0.19490,0.13593,0.50016,0.47934,1.38173,0.25028,0.79749,2.74803,0.33684,1.50395,0.17176,0.50027,0.22068,0.16912,0.15127,0.30002,0.10574,0.09906,0.58486,0.29294,0.322655
ENSG0000097,1.78586,0.66128,0.05357,1.89445,2.24005,0.56520,0.60090,0.93570,0.29183,0.13115,1.86196,0.34379,1.00044,0.16080,0.63987,1.44436,0.03145,0.65580,0.23896,0.29616,0.12068,0.48084,1.32155,0.31045,0.73256,1.70285,0.50537,0.25939,0.81693,0.13300,0.82658,1.83353,0.12410,0.70999,0.35762,0.20096,0.04993,1.46284,1.31132,0.83182,0.37647,0.36062,0.60773,0.15770,0.10744,0.07517,1.62847,0.86

In [9]:
sample = architecture.load_input('sample_tissues.csv')
validator = InputValidation(sample, gct)
sample = validator.run_all_validations()
sample

,Gene,TPM
0,ENSG000000,0.745360
1,ENSG000001,0.179731
2,ENSG000002,0.745843
3,ENSG000003,0.422376
4,ENSG000004,0.343085
...,...,...
95,ENSG0000095,0.354193
96,ENSG0000096,0.549967
97,ENSG0000097,0.385978
98,ENSG0000098,0.362802


In [10]:
corr = math_logic.calculate_spearman(sample, gct)
ranked_tissues = math_logic.rank_matches(corr)
visualization.plot_correlations(ranked_tissues, ".") #automatically outputs named png, can specify filepath
